Data is cleaned at the start. We remove duplicates, missing values, and outliers so the models learn from valid data.

In [9]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
%matplotlib inline

In [10]:
raw_big5_data = pd.read_csv('drafts/datasets/big5-dataset.csv')
row_count_before_cleaning = len(raw_big5_data)

deduplicated_big5_data = raw_big5_data.drop_duplicates().copy()
row_count_after_deduplication = len(deduplicated_big5_data)

base_clean_big5_data = deduplicated_big5_data.dropna().copy()
row_count_after_missing_value_removal = len(base_clean_big5_data)

print(f'Rows before cleaning: {row_count_before_cleaning}')
print(f'Rows after duplicate removal: {row_count_after_deduplication}')
print(f'Rows after missing-value removal: {row_count_after_missing_value_removal}')

base_clean_big5_data.head()

Rows before cleaning: 19719
Rows after duplicate removal: 19719
Rows after missing-value removal: 19710


,race,age,engnat,gender,hand,source,country,E1,E2,E3,...,O1,O2,O3,O4,O5,O6,O7,O8,O9,O10
0,3,53,1,1,1,1,US,4,2,5,...,4,1,3,1,5,1,4,2,5,5
1,13,46,1,2,1,1,US,2,2,3,...,3,3,3,3,2,3,3,1,3,2
2,1,14,2,2,1,1,PK,5,1,1,...,4,5,5,1,5,1,5,5,5,5
3,3,19,2,2,1,1,RO,2,5,2,...,4,3,5,2,4,2,5,2,5,5
4,11,25,2,2,1,2,US,3,1,3,...,3,1,1,1,3,1,3,1,5,3


In [11]:
big5_with_trait_totals = base_clean_big5_data.copy()

trait_item_prefix_by_total_column = {
    'A_total': '^A\d',
    'O_total': '^O\d',
    'C_total': '^C\d',
    'E_total': '^E\d',
    'N_total': '^N\d'
}

reverse_keyed_items = {
    'E_total': ['E2', 'E4', 'E6', 'E8', 'E10'],
    'A_total': ['A1', 'A3', 'A5', 'A7'],
    'C_total': ['C2', 'C4', 'C6', 'C8'],
    'N_total': ['N1', 'N3', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10'],
    'O_total': ['O2', 'O4', 'O6']
}

for trait_total_column, reverse_items in reverse_keyed_items.items():
    for item_col in reverse_items:
        if item_col in big5_with_trait_totals.columns:
            big5_with_trait_totals[item_col] = 6 - pd.to_numeric(
                big5_with_trait_totals[item_col], errors='coerce'
            )

for trait_total_column, item_regex in trait_item_prefix_by_total_column.items():
    trait_item_columns = big5_with_trait_totals.filter(regex=item_regex)
    big5_with_trait_totals[trait_total_column] = trait_item_columns.sum(axis=1)

trait_total_columns = list(trait_item_prefix_by_total_column.keys())
big5_with_trait_totals[trait_total_columns].describe()

<>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not w

,A_total,O_total,C_total,E_total,N_total
count,19710.000000,19710.000000,19710.000000,19710.000000,19710.000000
mean,38.444597,39.087722,33.472400,30.114409,29.030999
std,7.146748,6.256331,7.306113,9.223418,8.617323
min,10.000000,10.000000,10.000000,10.000000,10.000000
25%,34.000000,35.000000,28.000000,23.000000,23.000000
50%,39.000000,40.000000,34.000000,30.000000,29.000000
75%,44.000000,44.000000,39.000000,37.000000,35.000000
max,50.000000,50.000000,50.000000,50.000000,50.000000


In [12]:
trait_totals_for_cleaning = big5_with_trait_totals[trait_total_columns].dropna().copy()

lower_tail_quantile = 0.01
upper_tail_quantile = 0.975

lower_tail_bounds = trait_totals_for_cleaning.quantile(lower_tail_quantile)
upper_tail_bounds = trait_totals_for_cleaning.quantile(upper_tail_quantile)

is_above_lower_bounds = trait_totals_for_cleaning.ge(lower_tail_bounds, axis=1)
is_below_upper_bounds = trait_totals_for_cleaning.le(upper_tail_bounds, axis=1)
is_within_strict_trait_bounds = (is_above_lower_bounds & is_below_upper_bounds).all(axis=1)

quantile_filtered_totals = trait_totals_for_cleaning.loc[is_within_strict_trait_bounds].copy()

median_values = quantile_filtered_totals.median()
mad_values = (quantile_filtered_totals - median_values).abs().median()
robust_z_scores = 0.6745 * (quantile_filtered_totals - median_values) / mad_values
is_within_robust_z = robust_z_scores.abs().le(3.5).all(axis=1)

strict_clean_trait_totals = quantile_filtered_totals.loc[is_within_robust_z].copy()
final_row_mask = is_within_strict_trait_bounds & is_within_robust_z

rows_before_strict_cleaning = len(trait_totals_for_cleaning)
rows_after_strict_cleaning = len(strict_clean_trait_totals)
rows_removed_by_strict_cleaning = rows_before_strict_cleaning - rows_after_strict_cleaning
pct_removed_by_strict_cleaning = (rows_removed_by_strict_cleaning / rows_before_strict_cleaning) * 100

print(f'Rows before strict trait cleaning: {rows_before_strict_cleaning}')
print(f'Rows after strict trait cleaning: {rows_after_strict_cleaning}')
print(f'Rows removed by strict trait cleaning: {rows_removed_by_strict_cleaning} ({pct_removed_by_strict_cleaning:.2f}%)')

trait_scaler = StandardScaler()
standardized_trait_totals = pd.DataFrame(
    trait_scaler.fit_transform(strict_clean_trait_totals),
    columns=trait_total_columns,
    index=strict_clean_trait_totals.index
)

standardized_trait_totals.describe().T[['mean', 'std', 'min', 'max']]

# Save the cleaned dataset with totals and dashboard columns
dashboard_columns = ['age', 'gender'] + trait_total_columns
strict_clean_big5 = big5_with_trait_totals.loc[final_row_mask, dashboard_columns].copy()
strict_clean_big5.to_csv('Datasets/cleaned_big5_totals.csv', index=False)
print("Cleaned dataset saved to Datasets/cleaned_big5_totals.csv")

Rows before strict trait cleaning: 19710
Rows after strict trait cleaning: 17464
Rows removed by strict trait cleaning: 2246 (11.40%)
Cleaned dataset saved to Datasets/cleaned_big5_totals.csv
